In [2]:
import numpy as np
import pandas as pd
import rasterio as rio
from scipy.optimize import curve_fit
from matplotlib import pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
from pathlib import Path

In [3]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"

In [3]:
country_raster_1km_  = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [5]:
global_building_minerals = rio.open(RESULTS / 'features' / 'building-mineral-stock-1km.tif').read(1)
population_2020 = rio.open(RESULTS / 'population-rasters' / 'population-2020-1km.tif').read(1)
GDP_2020 = rio.open(RESULTS / 'GDP-rasters' / 'GDP-2020-1km.tif').read(1)

In [6]:
mask = (population_2020 > 0) & (GDP_2020 > 0) & (global_building_minerals > 0)

In [7]:
S = global_building_minerals[mask]
S_pc = global_building_minerals[mask] / population_2020[mask]      # t/cap
GDP_pc = GDP_2020[mask] / population_2020[mask]
density = population_2020[mask]

In [8]:
density_bins = [1, 10, 50, 100, 500, 1000, 5000, 10000, np.inf]
density_labels = [
    "1-10", "10-50", "50-100", "100-500",
    "500-1000", "1000-5000", "5000-10000", "≥10000"
]

In [9]:
gdp_bins = [0, 1000, 2000, 3000, 5000, 8000, 12000, 20000, 30000, 45000, 60000, 90000, np.inf]
gdp_labels = [
    "<1000",
    "1k-2k",
    "2k-3k",
    "3k-5k",
    "5k-8k",
    "8k-12k",
    "12k-20k",
    "20k-30k",
    "30k-45k",
    "45k-60k",
    "60k-90k",
    "≥90k"
]

In [10]:
df = pd.DataFrame({"GDP_pc": GDP_pc, "density": density, "S_pc": S_pc})
df["gdp_bin"] = pd.cut(df["GDP_pc"], bins=gdp_bins, labels=gdp_labels, right=False)
df["den_bin"] = pd.cut(df["density"], bins=density_bins, labels=density_labels, right=False)

In [11]:
df_2 = pd.DataFrame({"GDP_pc": GDP_pc, "density": density, "S": S})
df_2["gdp_bin"] = pd.cut(df_2["GDP_pc"], bins=gdp_bins, labels=gdp_labels, right=False)
df_2["den_bin"] = pd.cut(df_2["density"], bins=density_bins, labels=density_labels, right=False)

In [12]:
print(df["gdp_bin"].value_counts(dropna=False).sort_index())
print(df_2["gdp_bin"].value_counts(dropna=False).sort_index())

gdp_bin
<1000      18174093
1k-2k      12448498
2k-3k       5351837
3k-5k       3229195
5k-8k       1361179
8k-12k       518907
12k-20k      547347
20k-30k      246833
30k-45k      256853
45k-60k       89237
60k-90k       79742
≥90k          74696
Name: count, dtype: int64
gdp_bin
<1000      18174093
1k-2k      12448498
2k-3k       5351837
3k-5k       3229195
5k-8k       1361179
8k-12k       518907
12k-20k      547347
20k-30k      246833
30k-45k      256853
45k-60k       89237
60k-90k       79742
≥90k          74696
Name: count, dtype: int64


In [13]:
df_valid = df.dropna(subset=["den_bin", "gdp_bin"])
df_valid_2 = df_2.dropna(subset=["den_bin", "gdp_bin"])

In [ ]:
summary_1 = df_valid.groupby(["den_bin", "gdp_bin"]).agg(
    median=("S_pc", "median"),
    p25=("S_pc", lambda x: np.percentile(x, 25)),
    p75=("S_pc", lambda x: np.percentile(x, 75)),
    p5=("S_pc", lambda x: np.percentile(x, 5)),
    p95=("S_pc", lambda x: np.percentile(x, 95)),
    count=("S_pc", "size")
).reset_index()

In [15]:
summary_2 = df_valid_2.groupby(["den_bin", "gdp_bin"]).agg(
    median=("S", "median"),
    p25=("S", lambda x: np.percentile(x, 25)),
    p75=("S", lambda x: np.percentile(x, 75)),
    p5=("S", lambda x: np.percentile(x, 5)),
    p95=("S", lambda x: np.percentile(x, 95)),
    count=("S", "size")
).reset_index()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_68188\142293403.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary_2 = df_valid_2.groupby(["den_bin", "gdp_bin"]).agg(


In [22]:
summary_1.to_csv(RESULTS / 'bins-summary-1.csv', index=False)
summary_2.to_csv(RESULTS / 'bins-summary-2.csv', index=False)

In [ ]:
def determine_future_stock(year, SSP):
    if year == 2020:
        POP_t = rio.open(RESULTS / 'population-rasters' / f'population-{year}-1km.tif').read(1)
        GDP_t = rio.open(RESULTS / 'GDP-rasters' / f'GDP-{year}-1km.tif').read(1)
        
    else:
        POP_t = rio.open(RESULTS / 'population-rasters' / SSP / f'population-{year}-{SSP}-1km.tif').read(1)
        GDP_t = rio.open(RESULTS / 'GDP-rasters' / f'GDP-{year}-{SSP}-1km.tif').read(1)

    valid = (POP_t > 0) & (GDP_t > 0)

    density_t = np.full(POP_t.shape, np.nan, dtype=np.float32)
    GDP_pc_t = np.full(GDP_t.shape, np.nan, dtype=np.float32)

    density_t[valid] = POP_t[valid]
    GDP_pc_t[valid] = GDP_t[valid] / POP_t[valid]

    den_id = np.digitize(density_t, density_bins, right=True).astype(np.int8)
    gdp_id = np.digitize(GDP_pc_t, gdp_bins, right=True).astype(np.int8)

    summary_1["den_id"] = summary_1["den_bin"].apply(lambda x: density_labels.index(x) + 1)
    summary_1["gdp_id"] = summary_1["gdp_bin"].apply(lambda x: gdp_labels.index(x) + 1)
    summary_2["den_id"] = summary_2["den_bin"].apply(lambda x: density_labels.index(x) + 1)
    summary_2["gdp_id"] = summary_2["gdp_bin"].apply(lambda x: gdp_labels.index(x) + 1)

    S_pc_t = np.zeros(shape=global_building_minerals.shape, dtype=np.float32)
    S_t = np.zeros(shape=global_building_minerals.shape, dtype=np.float32)


    pairs = [(i, j) for i in range(1, len(density_bins)) for j in range(1, len(gdp_bins))]
    for i, j in tqdm(pairs, desc="Processing bins"):
        a_target = summary_1[(summary_1['den_id'] == i) & (summary_1['gdp_id'] == j)]['median'].values[0]
        a_cap = summary_2[(summary_2['den_id'] == i) & (summary_2['gdp_id'] == j)]['median'].values[0]
        a_mask = (den_id == i) & (gdp_id == j)
        S_pc_t[a_mask] = a_target
        gg = S_pc_t[a_mask] * POP_t[a_mask]
        hh = np.where(gg > a_cap, a_cap, gg)
        S_t[a_mask] = np.maximum(hh, global_building_minerals[a_mask])
        # print(i, j, a_target, a_cap)

    return S_t

In [ ]:
def save_results(data, year, SSP):
    with rio.open(
        RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{year}-{SSP}.tif', 
        'w', 
        driver='GTiff',
        height=country_raster_1km_.shape[0],
        width=country_raster_1km_.shape[1],
        count=1,
        dtype=data.dtype,
        crs=country_raster_1km_.crs,
        transform=country_raster_1km_.transform, 
        compress='lzw',
    ) as dst:
        dst.write(data, 1)

In [ ]:
for SSP in tqdm(['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']):
    for t in tqdm(range(2020, 2055, 5)):
        # print(t)
        S_t_SSP = determine_future_stock(t, SSP)
        print(np.sum(S_t_SSP) / 1e9)
        save_results(S_t_SSP, t, SSP)


  0%|          | 0/2 [00:00<?, ?it/s]
































































































Processing bins: 100%|██████████| 96/96 [02:04<00:00,  1.29s/it]


464.53607



































































































Processing bins: 100%|██████████| 96/96 [02:01<00:00,  1.27s/it]


479.8251



































































































Processing bins: 100%|██████████| 96/96 [02:02<00:00,  1.27s/it]


...


































































































Processing bins: 100%|██████████| 96/96 [02:01<00:00,  1.27s/it]


642.41626



































































































Processing bins: 100%|██████████| 96/96 [02:01<00:00,  1.27s/it]


691.49146


100%|██████████| 2/2 [29:20<00:00, 880.03s/it]


In [ ]:
S_2050_SSP1 = determine_future_stock(2050, 'SSP1')
S_2050_SSP2 = determine_future_stock(2050, 'SSP2')
S_2050_SSP3 = determine_future_stock(2050, 'SSP3')
S_2050_SSP4 = determine_future_stock(2050, 'SSP4')
S_2050_SSP5 = determine_future_stock(2050, 'SSP5')

Processing bins: 100%|██████████| 96/96 [02:00<00:00,  1.26s/it]


In [ ]:
print(np.sum(global_building_minerals) / 1e9)
print(np.sum(S_2050_SSP1) / 1e9)
print(np.sum(S_2050_SSP2) / 1e9)
print(np.sum(S_2050_SSP3) / 1e9)
print(np.sum(S_2050_SSP4) / 1e9)
print(np.sum(S_2050_SSP5) / 1e9)

401.41298156558753
636.7669
585.03406
532.57324
548.78827
691.49146


interpolate annual growth

In [6]:
df_cement_production = pd.read_csv(DATA / 'production-142.csv', index_col=0)
economies_142 = df_cement_production.columns.tolist()
country_pseduo_values = pd.read_csv(DATA / 'country-pseudo-values.csv')
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')
economy_raster_1km_ = rio.open(RESULTS / 'economy-rasters' / 'economy-raster-eck4-1km.tif')
pseduo_codes_economies_142 = pd.read_csv(RESULTS / 'pseudo-codes-economies-142.csv')

In [ ]:
country_raster_1km = country_raster_1km_.read(1)
economy_raster_1km = economy_raster_1km_.read(1)

In [ ]:
results_ = []
for SSP in ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']:
    for t in range(2020, 2055, 5):
        input_path = RESULTS / 'building-mineral-rasters' / f'global-building-minerals-{t}-{SSP}.tif'
        global_building_raster = rio.open(input_path).read(1)

        for idx in tqdm(range(1, 143), desc=f'processing {t} for {SSP}'):   # 以后也可以换成国家列表
            economy = pseduo_codes_economies_142[pseduo_codes_economies_142['value'] == idx]['economies_142'].values[0]
            a_economy_total = np.nansum(global_building_raster[economy_raster_1km == idx])
            results_.append({
                'scenario': SSP,
                'year': t,
                'economy': economy,
                'value': a_economy_total
            })

results = pd.DataFrame(results_)

In [ ]:
results_full_series_ = []

for economy in tqdm(economies_142):
    economy_results_ = []
    for scenario in ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']:
        part_ = results[(results['scenario'] == scenario) & (results['economy'] == economy)].set_index('year')
        part = part_.reindex(range(2020, 2051))
        part['value'] = part['value'].interpolate()
        part['scenario'] = part['scenario'].ffill()
        part['economy'] = part['economy'].ffill()
        economy_results_.append(part)

    economy_results = pd.concat(economy_results_)
    results_full_series_.append(economy_results)

results_full_series = pd.concat(results_full_series_)

In [12]:
with pd.ExcelWriter(RESULTS / 'future-stock-growth.xlsx') as writer:
    for scenario in ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']:
        data = {}
        for economy in economies_142:
            a_slice = results_full_series[(results_full_series['economy'] == economy) & (results_full_series['scenario'] == scenario)].set_index('year')
            data[economy] = a_slice.loc[2022:, 'value'] / a_slice.loc[2022, 'value']

        df = pd.DataFrame(data, index=np.arange(2022, 2051))
        df.to_excel(writer, sheet_name=scenario)